In [ ]:
import gc
import torch

# 先尽量清理上一轮遗留的显存缓存，避免 notebook 之间互相影响。
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print('GPU cache cleared.')
else:
    print('CUDA not available, skip GPU cache clear.')


# 下一课：CartPole + Actor-Critic

前一课我们学了 `REINFORCE with Baseline`，已经同时拥有：
- 一个策略网络 `policy_net`
- 一个价值网络 `value_net`

这一课我们正式进入 `Actor-Critic`。

它的核心变化是：

**不再等整轮 episode 结束才更新，而是可以边与环境交互边更新。**


## 1. 为什么要从 REINFORCE 走到 Actor-Critic

REINFORCE 的优点是概念直观，但它有两个明显问题：
- 必须等一整轮结束后才能更新
- 方差比较大，训练容易抖

Actor-Critic 的想法是：
- `Actor` 负责学策略，决定动作概率
- `Critic` 负责学价值，评估当前动作/状态到底好不好

而且 Critic 不需要等整轮结束，它可以用一步 TD 误差更快地给 Actor 提供反馈。


## 2. Actor 和 Critic 各自是谁

在这一课里：

- `Actor`：策略网络，输出动作分布
- `Critic`：价值网络，输出状态价值 `V(s)`

它们的分工可以这样记：

- Actor 回答：“现在该怎么做？”
- Critic 回答：“你刚才做得怎么样？”

所以 Critic 就像评论员，Actor 像执行者。


## 3. Actor-Critic 最关键的信号：TD error

这节课最重要的一个量就是：

$\delta = r + \gamma V(s') - V(s)$

它叫 `TD error`。

你可以把它理解成：
- 当前状态本来估了一个分数 `V(s)`
- 但真正走一步之后，看起来应该更接近 `r + gamma * V(s')`
- 两者的差，就是 Critic 发现的“预测误差”

然后：
- Critic 用它来修正自己的价值估计
- Actor 用它来决定刚才那个动作值不值得鼓励


In [ ]:
import random
import warnings

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=3, suppress=True)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
def pick_device():
    if torch.cuda.is_available():
        try:
            _ = torch.zeros(1, device='cuda')
            return torch.device('cuda')
        except Exception as e:
            warnings.warn(f'检测到 CUDA，但当前环境无法真正使用 GPU，已回退到 CPU。原因: {e}')
    return torch.device('cpu')


device = pick_device()
print('当前设备:', device)
if torch.cuda.is_available():
    print('检测到 CUDA 设备:', torch.cuda.get_device_name(0))


In [ ]:
env = gym.make('CartPole-v1')
state, info = env.reset(seed=42)
print('初始状态:', state)
print('状态维度:', env.observation_space.shape[0])
print('动作数量:', env.action_space.n)


In [ ]:
def to_tensor(x, device):
    return torch.tensor(x, dtype=torch.float32, device=device)


class ActorNet(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )

    def forward(self, x):
        return self.net(x)


class CriticNet(nn.Module):
    def __init__(self, state_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x):
        return self.net(x)


## 4. 训练 Actor-Critic

这节课你要重点看训练循环里的这些步骤：

1. Actor 输出动作分布并采样动作
2. Critic 评估当前状态 `V(s)` 和下一状态 `V(s')`
3. 计算一步 TD error
4. Critic 用 TD target 学价值
5. Actor 用 TD error 作为 advantage 更新策略

这就是最基础的一步 Actor-Critic 主线。


In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

actor = ActorNet(state_dim, action_dim, hidden_dim=128).to(device)
critic = CriticNet(state_dim, hidden_dim=128).to(device)

actor_optimizer = optim.Adam(actor.parameters(), lr=3e-4)
critic_optimizer = optim.Adam(critic.parameters(), lr=1e-3)
critic_criterion = nn.MSELoss()

gamma = 0.99
episodes = 600
max_steps = 500
entropy_coef = 0.01

episode_rewards = []
actor_loss_history = []
critic_loss_history = []
td_error_history = []

for episode in range(episodes):
    state, info = env.reset()
    states = []
    next_states = []
    rewards = []
    dones = []
    log_probs = []
    entropies = []
    total_reward = 0.0

    for step in range(max_steps):
        state_tensor = to_tensor(state, device).unsqueeze(0)

        logits = actor(state_tensor)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        entropy = dist.entropy()

        next_state, reward, terminated, truncated, info = env.step(int(action.item()))
        done = terminated or truncated
        states.append(state)
        next_states.append(next_state)
        rewards.append(reward)
        dones.append(float(done))
        log_probs.append(log_prob)
        entropies.append(entropy)

        total_reward += reward
        state = next_state

        if done:
            break

    states_tensor = to_tensor(np.array(states), device)
    next_states_tensor = to_tensor(np.array(next_states), device)
    rewards_tensor = torch.tensor(rewards, dtype=torch.float32, device=device)
    dones_tensor = torch.tensor(dones, dtype=torch.float32, device=device)
    log_probs_tensor = torch.stack(log_probs).squeeze(-1)
    entropy_tensor = torch.stack(entropies).squeeze(-1)

    values = critic(states_tensor).squeeze(1)
    with torch.no_grad():
        next_values = critic(next_states_tensor).squeeze(1)
        td_targets = rewards_tensor + gamma * next_values * (1 - dones_tensor)

    advantages = td_targets - values

    # Actor 使用一步 TD advantage 做更新，并加入少量熵奖励保持探索。
    actor_loss = -(log_probs_tensor * advantages.detach()).mean() - entropy_coef * entropy_tensor.mean()

    # Critic 学会让自己的 V(s) 接近 TD target。
    critic_loss = critic_criterion(values, td_targets)

    actor_optimizer.zero_grad()
    actor_loss.backward()
    torch.nn.utils.clip_grad_norm_(actor.parameters(), max_norm=10.0)
    actor_optimizer.step()

    critic_optimizer.zero_grad()
    critic_loss.backward()
    torch.nn.utils.clip_grad_norm_(critic.parameters(), max_norm=10.0)
    critic_optimizer.step()

    actor_loss_history.append(float(actor_loss.item()))
    critic_loss_history.append(float(critic_loss.item()))
    td_error_history.append(float(advantages.mean().item()))
    episode_rewards.append(total_reward)

print('训练完成。')
print('最后 20 轮平均 reward:', round(float(np.mean(episode_rewards[-20:])), 2))


## 5. 看训练曲线

这节课我们会同时看：
- 每轮 reward
- actor loss
- critic loss
- TD error

因为现在是两个网络协作学习，而不是单一网络。


In [ ]:
window = 10
smoothed_rewards = np.convolve(episode_rewards, np.ones(window) / window, mode='valid')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(episode_rewards, color='#1f77b4', alpha=0.5, label='raw')
axes[0, 0].plot(range(window - 1, len(episode_rewards)), smoothed_rewards, color='#d62728', label='smoothed')
axes[0, 0].set_title('每轮 reward')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].legend()

axes[0, 1].plot(actor_loss_history, color='#2ca02c', alpha=0.7)
axes[0, 1].set_title('Actor Loss')
axes[0, 1].set_xlabel('Update Step')
axes[0, 1].set_ylabel('Loss')

axes[1, 0].plot(critic_loss_history, color='#ff7f0e', alpha=0.7)
axes[1, 0].set_title('Critic Loss')
axes[1, 0].set_xlabel('Update Step')
axes[1, 0].set_ylabel('Loss')

axes[1, 1].plot(td_error_history, color='#9467bd', alpha=0.7)
axes[1, 1].set_title('TD Error')
axes[1, 1].set_xlabel('Update Step')
axes[1, 1].set_ylabel('TD Error')

plt.tight_layout()
plt.show()


## 6. 多次测试平均表现

测试时和前几课一样，我们不按概率采样，而是直接选当前最优动作，方便看最终策略质量。


In [ ]:
test_env = gym.make('CartPole-v1')
test_rewards = []

actor.eval()
with torch.no_grad():
    for episode_idx in range(5):
        state, info = test_env.reset(seed=123 + episode_idx)
        test_reward = 0.0

        for step in range(500):
            state_tensor = to_tensor(state, device).unsqueeze(0)
            logits = actor(state_tensor)
            action = int(torch.argmax(logits, dim=1).item())

            state, reward, terminated, truncated, info = test_env.step(action)
            test_reward += reward
            if terminated or truncated:
                break

        test_rewards.append(test_reward)

print('测试 rewards:', test_rewards)
print('测试平均 reward:', round(float(np.mean(test_rewards)), 2))
test_env.close()


## 7. 这节课最值得记住什么

如果你想抓住这节课最核心的一句话，就是：

**Actor-Critic 用 Critic 提供一步 TD 反馈，让 Actor 不必等整轮结束就能开始学习。**

这就是它相比 REINFORCE 的关键进步。


## 8. 下一课最自然学什么

学完这一课后，最自然的下一步就是：
- `A2C / Advantage Actor-Critic`

因为你现在已经理解了：
- Actor
- Critic
- TD error

接下来就可以开始把 advantage 的处理做得更系统。
